In [2]:
from zk_normal import *

from __future__ import annotations

1. 密碼學核心工具 (Cryptography Helpers)
這些是與底層數學、雜湊相關的純函式，通常在論文的「演算法定義 (Algorithm Definitions)」章節會提到。

generate_voter_secret(): 產生 31 bytes 的隨機私鑰。

compute_poseidon_hash(inputs): 封裝 Poseidon 雜湊邏輯。

compute_identity_commitment(voter_id_hash, secret): 計算公開承諾。

encrypt_vote_content(vote_content, public_key): 使用公鑰加密選票。

2. 選民註冊階段 (Registration Phase)
對應前端生成金鑰與後端驗證資格的流程。

verify_voter_eligibility(election_id, voter_id_hash): 檢查選民資格與是否已註冊。

register_voter_commitment(election_id, voter_id_hash, commitment): 將選民的 Commitment 寫入資料庫。

3. 投票與證明階段 (Voting & Proving Phase)
對應前端產生 ZK Proof 與送出選票的流程。

generate_vote_proof(voter_id_hash, secret): 產生 Groth16 的零知識證明與公開信號。

cast_vote(election_id, encrypted_vote, proof, public_signals): 投遞選票的核心 API 進入點。

4. 驗證與計票階段 (Verification & Tallying Phase)
對應後端驗證 ZKP 以及最終開票的流程。

verify_zk_proof(proof, public_signals, vkey_path): 呼叫底層驗證庫檢查證明是否合法。

process_vote_transaction(vote_record): 處理沒收鑰匙與存入無記名選票的原子操作。

tally_election_results(election_id, private_key): 解密所有選票並統計最終結果。

In [3]:
class ZKVotingSystem:
    def __init__(self, vkey_path):
        self.vkey_path = vkey_path

    # 產生 31 bytes 的隨機私鑰。
    def generateVoterSecret(self) -> str:
        random_bytes = os.urandom(31)
        return random_bytes.hex()

    # 計算公開承諾
    def computeIdentityCommitment(self, voter, secret: str) -> str:
        """
        計算使用者的身分承諾 (Identity Commitment)
        透過呼叫 Node.js 確保與 Circom 電路的 Poseidon 參數 100% 一致
        """
        voter_id_int = str(int(voter.hashId, 16))
        secret_int = str(int(secret, 16))
        
        try:
            # 呼叫 Node.js 腳本
            cmd = f"node {FILE_PATH['commitment']} {voter_id_int} {secret_int}"
            result = subprocess.run(
                cmd, 
                shell=True, 
                capture_output=True, 
                text=True, 
                check=True
            )
            
            # 把 JS 印出來的字串去頭去尾 (去掉換行符號)
            commitment_str = result.stdout.strip()
            return commitment_str
            
        except subprocess.CalledProcessError as e:
            print(f"Error computing commitment: {e.stderr}")
            raise Exception("Failed to compute Identity Commitment")

    # 將選民的 Commitment 寫入資料庫

    def registerVoterCommitment(self, election: Election, voter_id_hash: str, commitment: str) -> bool:
        # 將選民與對應的 Commitment 寫入選民名冊，並初始化為未投票
        election.voter_registry[voter_id_hash] = {
            "commitment": commitment,
            "has_voted": False
        }
        return True

    # 產生 Groth16 的零知識證明與公開信號
    def generateVoteProof(self, voter, secret: str) -> dict:
        # 1. Convert inputs to stringified integers for snarkjs
        voter_id_int = str(int(voter.hashId, 16))
        secret_int = str(int(secret, 16))

        input_data = {
            "voter_id": voter_id_int,
            "secret": secret_int
        }

        # 2. Write input.json
        with open("input.json", "w") as f:
            json.dump(input_data, f)

        try:
            # 3. Generate Witness using WebAssembly
            cmd_witness = "node vote_js/generate_witness.js vote_js/vote.wasm input.json witness.wtns"
            subprocess.run(
                cmd_witness,
                shell=True,
                check=True,
                capture_output=True
            )

            # 4. Generate the real ZK Proof using the proving key
            cmd_prove = "snarkjs groth16 prove vote_0001.zkey witness.wtns proof.json public.json"
            subprocess.run(
                cmd_prove,
                shell=True,
                check=True,
                capture_output=True
            )

            # 5. Read the generated real proof and public signals
            with open("proof.json", "r") as f:
                real_proof = json.load(f)
            with open("public.json", "r") as f:
                real_public_signals = json.load(f)

            # Clean up temporary files (Optional but recommended)
            os.remove("input.json")
            os.remove("witness.wtns")

            return {
                "proof": real_proof,
                "public_signals": real_public_signals
            }

        except subprocess.CalledProcessError as e:
            print(f"Error generating proof: {e.stderr}")
            raise Exception("Failed to generate ZK Proof")

    # 投遞選票
    def castVote(self, election: Election, voter: User, ballot: Ballot) -> bool:

        # List check
        if voter.hashId not in election.voter_registry:
            raise ValueError("VOTER_NOT_REGISTERED")

        is_valid = self.verifyZKProof(ballot.proof, ballot.public_signals)
        if not is_valid:
            raise ValueError("ZK_VERIFICATION_FAILED")

        voter_record = election.voter_registry[voter.hashId]

        # double voting check
        if voter_record["has_voted"]:
            raise ValueError("ALREADY_VOTED")

        # 4. 驗證 ZKP 出示的承諾，是否與該選民當初註冊的承諾一致
        # 直接比對 ballot 裡的 commitment
        if voter_record["commitment"] != ballot.commitment:
            raise ValueError("COMMITMENT_MISMATCH")

        # 5. 在系統紀錄該選民已投票，並將不記名選票投入票匭
        voter_record["has_voted"] = True

        # 票匭只收加密選票，與身分完美切割
        election.vote_box.append(ballot.encrypted_vote)

        return True

    def verifyZKProof(self, proof: dict, public_signals: list) -> bool:
        """
        透過 snarkjs 驗證 Groth16 證明
        """
        vkey_path = self.vkey_path

        # 使用 tempfile 建立暫存資料夾，驗證完自動清理
        with tempfile.TemporaryDirectory() as temp_dir:
            proof_path = os.path.join(temp_dir, "proof.json")
            public_path = os.path.join(temp_dir, "public.json")

            # 將 Python 字典寫入臨時 JSON 檔案
            with open(proof_path, 'w') as f:
                json.dump(proof, f)
            with open(public_path, 'w') as f:
                json.dump(public_signals, f)

            try:
                # Use a single string command and shell=True for macOS compatibility
                cmd = f"snarkjs groth16 verify {vkey_path} {public_path} {proof_path}"
                
                result = subprocess.run(
                    cmd,
                    shell=True,
                    capture_output=True,
                    text=True,
                    check=True 
                )
                
                if "OK" in result.stdout:
                    return True
                else:
                    print(f"ZKP verification failed: {result.stdout}")
                    return False
                    
            except subprocess.CalledProcessError as e:
                print(f"[SnarkJS Error] Proof not eligible: {e.stderr}")
                return False

    def tally(self, election: Election, decrypt_fn) -> dict:
        """
        Tally the election results.
        :param election: The Election object containing the vote box.
        :param decrypt_fn: A function or service that can decrypt the vote_content.
        :return: A dictionary containing the results.
        """
        results = {
            "candidate_votes": {c_id: 0 for c_id in election.candidates},
            "blank_votes": 0,
            "total_votes": len(election.vote_box)
        }

        # Process each encrypted vote in the box
        for encrypted_vote in election.vote_box:
            try:
                # Decrypt the vote back to its original value (e.g., candidate ID)
                decrypted_val = decrypt_fn(encrypted_vote)

                # Check if it's a blank vote or a valid candidate ID
                if decrypted_val == Election.BLANK_VOTE:
                    results["blank_votes"] += 1
                elif decrypted_val in results["candidate_votes"]:
                    results["candidate_votes"][decrypted_val] += 1
                else:
                    # Optional: Handle cases where decrypted value is not in candidate list
                    print(
                        f"Warning: Decrypted value {decrypted_val} is not a valid candidate.")
                    results["blank_votes"] += 1

            except Exception as e:
                print(f"Error decrypting vote: {e}")
                results["blank_votes"] += 1

        return results

In [4]:
if __name__ == "__main__":
    # 1. Initialize system and election
    zk_sys = ZKVotingSystem(vkey_path=FILE_PATH['vkey'])
    # Candidates ID: 1, 2
    test_election = Election(name="TestElection", candidates=[1, 2])
    
    # 2. Mock reading user from CSV
    voters = readCsvData(FILE_PATH['csv'])
    
    print(f"--- Registration Phase ---")
    # Generate secret and register commitment
    
    for voter in voters:
        secret = zk_sys.generateVoterSecret()
        commitment = zk_sys.computeIdentityCommitment(voter, secret)
        zk_sys.registerVoterCommitment(test_election, voter.hashId, commitment)
        print(f"User {voter.id} registered with commitment: {commitment[:10]}...")
        
        print(f"\n--- Voting Phase ---")
        # User decides vote
        vote_choice = voter.userRandomVote(test_election)
        print(f"User decides to vote for: {vote_choice}")
        
        # Mock encryption (simulate frontend RSA encryption)
        mock_encrypted_vote = base64.b64encode(str(vote_choice).encode('utf-8')).decode('utf-8')
        
        # Generate ZK Proof
        zk_data = zk_sys.generateVoteProof(voter, secret)
        
        # Package into Ballot
        ballot = Ballot(
            encrypted_vote=mock_encrypted_vote,
            proof=zk_data["proof"],
            public_signals=zk_data["public_signals"]
        )
        print("Ballot packaged successfully.")
        
        print(f"\n--- Casting Vote ---")
        try:
            # Cast the ballot
            success = zk_sys.castVote(test_election, voter, ballot)
            if success:
                print("Vote casted successfully! Vote is in the box.")
        except Exception as e:
            print(f"Vote failed: {e}")
        
        print(f"\n--- Verification ---")
        print(f"Has voted status: {test_election.voter_registry[voter.hashId]['has_voted']}")
        print(f"Total votes in box: {len(test_election.vote_box)}")
        
    def mock_decrypt(enc_vote):
        return int(base64.b64decode(enc_vote.encode('utf-8')).decode('utf-8'))

    print("\n--- 開票階段 ---")
    results = zk_sys.tally(test_election, mock_decrypt)
    
    print(f"選舉名稱: {test_election.name}")
    print(f"總投票數: {results['total_votes']}")
    print(f"廢票數: {results['blank_votes']}")
    print("各候選人得票統計:")
    for c_id, count in results['candidate_votes'].items():
        print(f"  - 候選人 {c_id}: {count} 票")

--- Registration Phase ---
User S1242121 registered with commitment: 3123255665...

--- Voting Phase ---
User decides to vote for: 1
Ballot packaged successfully.

--- Casting Vote ---
Vote casted successfully! Vote is in the box.

--- Verification ---
Has voted status: True
Total votes in box: 1
User S1331116 registered with commitment: 8103745263...

--- Voting Phase ---
User decides to vote for: 2
Ballot packaged successfully.

--- Casting Vote ---
Vote casted successfully! Vote is in the box.

--- Verification ---
Has voted status: True
Total votes in box: 2
User M1452004 registered with commitment: 1114391206...

--- Voting Phase ---
User decides to vote for: 2
Ballot packaged successfully.

--- Casting Vote ---
Vote casted successfully! Vote is in the box.

--- Verification ---
Has voted status: True
Total votes in box: 3
User M1452019 registered with commitment: 7686160829...

--- Voting Phase ---
User decides to vote for: 1
Ballot packaged successfully.

--- Casting Vote ---
Vo